In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
sns.set_theme(font=["WenQuanYi Zen Hei"])
sns.set_style({'axes.unicode_minus': False})

print("环境通过，开始第一个机器学习项目")

In [ ]:
iris = datasets.load_iris()
x = iris.data
y = iris.target
target_names = iris.target_names

print(f"数据形状：特征 {x.shape}，标签 {y.shape}")
print(f"特征名称：{iris.feature_names}")
print(f"标签类别：{target_names}")
print(f"样本分布：\n{pd.Series(y).value_counts().to_dict()}")

df = pd.DataFrame(x, columns=iris.feature_names)
df['species'] = [target_names[i] for i in y]
print("\n 前五行数据：")
print(df.head())

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, feature in enumerate(iris.feature_names):
    ax = axes[idx//2, idx % 2]
    for species_idx, species_name in enumerate(target_names):
        ax.hist(x[y == species_idx, idx], alpha=0.7,
                label=species_name, bins=15)
    ax.set_xlabel(feature)
    ax.set_ylabel("频数")
    ax.legend()
plt.suptitle("鸢尾花各特征分布", fontsize=16)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
for species_idx, species_name in enumerate(target_names):
    plt.scatter(x[y == species_idx, 0],
                x[y == species_idx, 1],
                label=species_name,
                alpha=0.7, s=50)
plt.xlabel("花萼长度(cm)")
plt.ylabel("花萼宽度(cm)")
plt.title("鸢尾花数据分布散点图")
plt.legend()
plt.show()

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
print("标准化后的特征（前五行）：")
print(pd.DataFrame(x_scaled[:5], columns=iris.feature_names).round(3))

x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"\n 数据集划分：")
print(f"训练集：{x_train.shape[0]}个样本")
print(f"测试集：{x_test.shape[0]}个样本")

In [ ]:
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(x_train, y_train)

print("模型训练完成！")
print(f"模型参数：{model.coef_.shape}")
print(f"模型偏置：{model.intercept_}")
print(f"模型在训练集上的准确率：{model.score(x_train, y_train):.3f}")
print(f"模型在测试集上的准确率：{model.score(x_test, y_test):.3f}")

In [ ]:
y_pred=model.predict(x_test)
y_pred_proba=model.predict_proba(x_test)
print("详细分类报告：")
print(classification_report(y_test,y_pred,target_names=target_names))

cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(10,10))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=target_names,yticklabels=target_names)
plt.xlabel("预测标签")
plt.ylabel("真实标签")
plt.title("混淆矩阵")
plt.show()

In [ ]:
def plot_decision_boundary(model, x, y, feature_names, target_names):
    feature_idx = [0, 2]
    x_2d = x[:, feature_idx]

    model_2d = LogisticRegression(random_state=42, max_iter=200)
    model_2d.fit(x_2d, y)

    x_min, x_max = x_2d[:, 0].min()-0.5, x_2d[:, 0].max()+0.5
    y_min, y_max = x_2d[:, 1].min()-0.5, x_2d[:, 1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))

    z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    z = z.reshape(xx.shape)

    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, z, alpha=0.3, cmap='viridis')

    scatter = plt.scatter(x_2d[:, 0], x_2d[:, 1], c=y,
                          edgecolors='black', s=50, cmap='viridis')
    plt.xlabel(feature_names[feature_idx[0]])
    plt.ylabel(feature_names[feature_idx[1]])
    plt.title('决策边界可视化')

    legend_labels = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.viridis(
        i/3), markersize=10, label=target_names[i]) for i in range(3)]
    plt.legend(handles=legend_labels,title='鸢尾花类型')
    plt.show()
    
plot_decision_boundary(model,x_scaled,y,iris.feature_names,iris.target_names)

In [ ]:
import joblib

joblib.dump(model,'../models/iris_classifier.pkl')
joblib.dump(scaler,'../models/iris_scaler.pkl')
print('模型已经保存为../models/iris_classifier.pkl和../models/iris_scaler.pkl')